# 04 — Fairness e fedeltà delle spiegazioni

Domande: il modello tratta i gruppi in modo diverso, anche senza vedere gli attributi sensibili? Le spiegazioni in linguaggio naturale dicono la verità sul modello?

Prima di eseguire: `python -m credit_xai.pipeline` dalla cartella del progetto.

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import json
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

from credit_xai import config
from credit_xai.data import load_processed
from credit_xai.features import get_X_y
from credit_xai.model import CreditModel

sns.set_theme(style='whitegrid')
train, test = load_processed('train'), load_processed('test')
X_test, y_test = get_X_y(test)
model = CreditModel.load()

from credit_xai.fairness import audit, mitigate

## 1. Audit per gruppo

In [ ]:
results = audit(model, test)
for attr, res in results.items():
    print(f"\n{attr}: DP diff {res['demographic_parity_difference']} | "
          f"EO diff {res['equalized_odds_difference']} | DI ratio {res['disparate_impact_ratio']}")
    display(pd.DataFrame(res['by_group']))

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for ax, (attr, res) in zip(axes, results.items()):
    df = pd.DataFrame(res['by_group']).set_index(attr)
    df[['approval_rate', 'false_rejection_rate']].plot.bar(ax=ax, rot=0)
    ax.set_title(attr); ax.set_ylim(0, 1)
plt.tight_layout()

## 2. Proxy: l'età entra nel modello?

L'età è una feature. Quanto pesa nelle decisioni?

In [ ]:
from credit_xai.explain.shap_explainer import ShapExplainer
sv = ShapExplainer(model).shap_values(test)
pd.DataFrame({'age_group': test['age_group'], 'shap_età': sv['age']})\
  .groupby('age_group')['shap_età'].mean().plot.bar(rot=0, title='Contributo medio dell\'età al rischio')

## 3. Mitigazione: soglie per gruppo (ThresholdOptimizer)

In [ ]:
pd.Series(mitigate(model, train, test, attribute='age_group'))

## 4. Fedeltà delle spiegazioni

Risultati di `python -m credit_xai.evaluation.faithfulness`. Per confrontare un LLM con il template, imposta `LLM_PROVIDER` nel file `.env` e rilancia.

In [ ]:
faith = config.REPORTS_DIR / 'faithfulness.json'
pd.Series(json.loads(faith.read_text())) if faith.exists() else 'Esegui prima la valutazione'

In [ ]:
cases_file = config.REPORTS_DIR / 'faithfulness_cases.jsonl'
cases = [json.loads(l) for l in cases_file.read_text(encoding='utf-8').splitlines()]
failed = [c for c in cases if not c['check']['faithful']]
print(f'{len(failed)} spiegazioni non fedeli su {len(cases)}')
for c in failed[:3]:
    print('\n', c['narration']['text'], '\n', c['check'])

## Note

- Gruppi con disparità: …
- Effetto della mitigazione: …
- Tasso di fedeltà template vs LLM: …